# Modelo SARIMA para predecir número de viajes

In [1]:
import pickle
import pandas as pd
import numpy as np

In [3]:
with open("../../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)

In [5]:
df_model = df_data.drop(columns=[
    "ride_id",
    "ended_at",
    "time_hms_ms",
    "member_casual",
    "start_station_id",
    "end_station_id",
    "month",
    "day",
    "temperature",
    "wind_speed",
    "precipitation",
    "relative_humidity",
    "snow_depth",
    "hour_float"
])

# Se agrupan los datos por viajes cada 30 minutos y entre estaciones

In [6]:
# Se redonde al minuto mas cercano
df_data["started_minute"] = df_data["started_at"].dt.round("30min")

In [7]:
df_agg = df_data.groupby([
    "start_station_idx",
    "end_station_idx",
    "started_minute"
]).agg(
    n_viajes=("ride_id", "count"),
    year=("year", "first"),
    temp_std=("temp_std", "first"),
    wind_std=("wind_std", "first"),
    rel_humidity_std=("rel_humidity_std", "first"),
    precipitation_std=("precipitation_std", "first"),
    snow_depth_std=("snow_depth_std", "first"),
    hour_sin=("hour_sin", "first"),
    hour_cos=("hour_cos", "first"),
    month_sin=("month_sin", "first"),
    month_cos=("month_cos", "first"),
    event=("event", "any"),  # True si al menos un dato es true
    normal_day=("day_type_Normal", "any"),  # True si al menos un dato es true
    weekend_day=("day_type_Weekend", "any"),  # True si al menos un dato es true
    holiday_day=("day_type_Holiday", "any"),  # True si al menos un dato es true
    #member_casual=("member_casual_bool", "any"),  # True si al menos un dato es true
    #classic_bike=("rideable_type_classic_bike", "any"),  # True si al menos un dato es true
    #docked_bike=("rideable_type_docked_bike", "any"),  # True si al menos un dato es true
    #electric_bike=("rideable_type_electric_bike", "any"),  # True si al menos un dato es true
    #duration_min_mean=("duration_min", "mean"),
).reset_index()

In [8]:
df_agg['n_viajes'].value_counts(normalize=False)


n_viajes
1     7597015
2      711947
3       97226
4       30005
5        8521
6        3108
7        1258
8         474
9         233
10         92
11         57
12         26
13         14
14          6
16          3
19          1
18          1
17          1
Name: count, dtype: int64

In [9]:
df_agg.dtypes

start_station_idx             int64
end_station_idx               int64
started_minute       datetime64[ns]
n_viajes                      int64
year                          int64
temp_std                    float64
wind_std                    float64
rel_humidity_std            float64
precipitation_std           float64
snow_depth_std              float64
hour_sin                    float64
hour_cos                    float64
month_sin                   float64
month_cos                   float64
event                          bool
normal_day                     bool
weekend_day                    bool
holiday_day                    bool
dtype: object

# Construcción del modelo SARIMA

In [ ]:
# Se definen los parametros 
p = None
d = None
q = None

P = None
D = None
Q = None

m = 12  # estacionalidad anual (12 meses)

# El coste de recursos y computacional para gestionar las combinaciones de 1912 estaciones es inviable, serían 3.658.000 combinaciones, es decir, 3.658.000 predicciones